<a href="https://colab.research.google.com/github/khaliqtaimoor6-codes/Machine-Learning-Practice/blob/Ridge-and-Lasso/Ridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [104]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,SGDRegressor
from sklearn.metrics import r2_score,mean_squared_error

In [105]:
data=load_diabetes()
print(data.DESCR)

.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measure of disease progression one year after baseline

:Attribute Information:
    - age     age in years
    - sex
    - bmi     body mass index
    - bp      average blood pressure
    - s1      tc, total serum cholesterol
    - s2      ldl, low-density lipoproteins
    - s3      hdl, high-density lipoproteins
    - s4      tch, total cholesterol / HDL
    - s5      ltg, possibly log of serum triglycerides level
    - s6      glu, blood sugar level

Note: Each of these 10 feature variables have bee

In [106]:
X=data.data
y=data.target

In [107]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=2)


In [108]:
lr=LinearRegression()
lr.fit(X_train,y_train)
y_pred=lr.predict(X_test)
print(r2_score(y_test,y_pred))
print(np.sqrt(mean_squared_error(y_test,y_pred)))

0.4399338661568968
55.627840795469155


In [109]:
from sklearn.linear_model import Ridge,Lasso
r=Ridge(alpha=0.01)
r.fit(X_train,y_train)
y_pred=r.predict(X_test)
print(r2_score(y_test,y_pred))
print(np.sqrt(mean_squared_error(y_test,y_pred)))#score slightly better than LR

0.4439392894728016
55.428567137568024


In [114]:

from sklearn.linear_model import Ridge

# Instantiate the Ridge model
# The 'alpha' parameter here is equivalent to the lambda_param or alpha you've been using for regularization strength.
# You can experiment with different alpha values.
ridge_sklearn = Ridge(alpha=0.1) # Using alpha=0.1 as an example

# Fit the model to the training data
ridge_sklearn.fit(X_train, y_train)

# Make predictions on the test set
y_pred_ridge_sklearn = ridge_sklearn.predict(X_test)

# Evaluate the model
print("R2 Score (sklearn Ridge):", r2_score(y_test, y_pred_ridge_sklearn))
print("RMSE (sklearn Ridge):", np.sqrt(mean_squared_error(y_test, y_pred_ridge_sklearn)))


R2 Score (sklearn Ridge): 0.45199494197195456
RMSE (sklearn Ridge): 55.02560551161431


### Ridge Regression with Gradient Descent

Ridge regression adds an L2 regularization term to the linear regression cost function. This term penalizes large coefficients, effectively shrinking them towards zero and reducing the model's complexity, which helps prevent overfitting.

The cost function for Ridge Regression is:
$$ J(w, b) = \frac{1}{2m} \sum_{i=1}^{m} (y^{(i)} - (w^T x^{(i)} + b))^2 + \lambda \sum_{j=1}^{n} w_j^2 $$

Where:
- $m$ is the number of training examples.
- $n$ is the number of features.
- $w$ is the vector of weights (coefficients).
- $b$ is the bias term.
- $\lambda$ is the regularization parameter, controlling the strength of the penalty.
- $x^{(i)}$ and $y^{(i)}$ are the $i$-th training example and its target value.

To minimize this cost function using Gradient Descent, we need to find the partial derivatives with respect to $w$ and $b$:

**Derivative with respect to weights ($w_j$):**
$$ \frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} ( (w^T x^{(i)} + b) - y^{(i)} ) x_j^{(i)} + \frac{\lambda}{m} w_j $$

**Derivative with respect to bias ($b$):**
$$ \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} ( (w^T x^{(i)} + b) - y^{(i)} ) $$

The update rules for Gradient Descent are then:
$$ w_j = w_j - \alpha \frac{\partial J}{\partial w_j} $$
$$ b = b - \alpha \frac{\partial J}{\partial b} $$

Where $\alpha$ is the learning rate.

In [110]:
class MeraRidgeGD:

  def __init__(self,epochs,learning_rate,alpha):
    self.learning_rate = learning_rate
    self.epochs = epochs
    self.alpha = alpha
    self.coef_ = None
    self.intercept_ = None

  def fit(self,X_train,y_train):
    self.coef_=np.ones(X_train.shape[1])
    self.intercept_=0
    theta=np.insert(self.coef_,0,self.intercept_)
    X_train=np.insert(X_train,0,1,axis=1)

    for i in range(self.epochs):
       theta_der = np.dot(X_train.T,X_train).dot(theta) - np.dot(X_train.T,y_train) + self.alpha*theta
       theta = theta - self.learning_rate*theta_der

    self.coef_ = theta[1:]
    self.intercept_ = theta[0]

  def predict(self,X_test):
      return np.dot(X_test,self.coef_) + self.intercept_

In [111]:
reg = MeraRidgeGD(epochs=500,learning_rate=0.005,alpha=0.001)
reg.fit(X_train,y_train)

In [112]:


y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.45395431712097367
[  19.50919039 -162.92602513  478.95477998  317.86376108  -34.07709121
 -108.63608801 -193.66871805  106.94769192  437.10746813  103.57606041]
152.03121813717044


As you can see, using `sklearn.linear_model.Ridge` is much more concise. It automatically handles the optimization (often using a closed-form solution or highly optimized numerical solvers, which are usually faster and more precise than a basic Gradient Descent implementation). You mainly need to choose the `alpha` (regularization strength) parameter.

### Ridge Regression using `sklearn.linear_model.SGDRegressor`

This method uses Stochastic Gradient Descent (SGD) to optimize the Ridge Regression objective function. `penalty='l2'` ensures that L2 regularization is applied, which is the core of Ridge Regression.

In [116]:
# Import the SGDRegressor
from sklearn.linear_model import SGDRegressor

# Instantiate SGDRegressor for Ridge Regression
# penalty='l2' specifies Ridge (L2 regularization)
# alpha is the regularization strength (similar to lambda)
# max_iter is the number of passes over the training data (epochs)
# eta0 is the initial learning rate
# random_state for reproducibility
sgd_ridge_model = SGDRegressor(penalty='l2', alpha=0.001, max_iter=100000, eta0=0.01, random_state=42)

# Fit the model to the training data
sgd_ridge_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_sgd_ridge = sgd_ridge_model.predict(X_test)

# Evaluate the model
print("RMSE (SGDRegressor Ridge):", np.sqrt(mean_squared_error(y_test, y_pred_sgd_ridge)))
print("R2 Score :",r2_score(y_test,y_pred_sgd_ridge))

RMSE (SGDRegressor Ridge): 55.33029632377312
R2 Score : 0.4459092519296439
